# Player Profile Clustering

Objective: group players within each position into clusters based on
playing style (performance metrics), to identify distinct player profiles
beyond traditional position labels.

In [13]:
import pandas as pd
from sklearn.cluster import KMeans

## Loading scored data

In [14]:
df = pd.read_csv("../data/processed/players_scored_2024.csv")

## Preparing features for clustering

In [15]:
metrics_to_normalize = ['goals', 'assists', 'xG', 'xAG', 'progressive_passes',
                         'progressive_carries', 'dribbles_successful', 'shot_creating_actions',
                         'goal_creating_actions', 'tackles', 'blocks']

In [16]:
percentile_columns = [f'{metric}_per90_percentile' for metric in metrics_to_normalize]
percentile_columns

['goals_per90_percentile',
 'assists_per90_percentile',
 'xG_per90_percentile',
 'xAG_per90_percentile',
 'progressive_passes_per90_percentile',
 'progressive_carries_per90_percentile',
 'dribbles_successful_per90_percentile',
 'shot_creating_actions_per90_percentile',
 'goal_creating_actions_per90_percentile',
 'tackles_per90_percentile',
 'blocks_per90_percentile']

## Clustering example: Centre-Forward

In [17]:
df_cf = df[df['position_group'] == 'Centre-Forward'].copy()
df_cf[percentile_columns].describe()

,goals_per90_percentile,assists_per90_percentile,xG_per90_percentile,xAG_per90_percentile,progressive_passes_per90_percentile,progressive_carries_per90_percentile,dribbles_successful_per90_percentile,shot_creating_actions_per90_percentile,goal_creating_actions_per90_percentile,tackles_per90_percentile,blocks_per90_percentile
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810
std,0.295468,0.295084,0.295468,0.295468,0.295468,0.295468,0.295468,0.295468,0.295084,0.295468,0.295468
min,0.047619,0.095238,0.047619,0.047619,0.047619,0.047619,0.047619,0.047619,0.095238,0.047619,0.047619
25%,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714,0.285714
50%,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810,0.523810
75%,0.761905,0.761905,0.761905,0.761905,0.761905,0.761905,0.761905,0.761905,0.761905,0.761905,0.761905
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [18]:
inertias = []
k_range = range(2, 8)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_cf[percentile_columns])
    inertias.append(kmeans.inertia_)

inertias

[14.089053803339517,
 12.008422416585685,
 10.144843969333767,
 8.755857898715043,
 7.821239606953894,
 6.779969765684051]

In [19]:
df_cf.shape[0]

21

In [20]:
kmeans_cf = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cf['cluster'] = kmeans_cf.fit_predict(df_cf[percentile_columns])
df_cf[['player_name', 'cluster']].sort_values('cluster')

,player_name,cluster
16,Hulk,0
17,Martin Braithwaite,0
18,Tiquinho Soares,0
43,Luciano,0
75,Alerrandro,0
93,Pedro,0
107,Yuri Alberto,0
8,Germán Cano,1
61,Pablo Vegetti,1
54,Jonathan Calleri,1


In [21]:
cluster_profiles = df_cf.groupby('cluster')[percentile_columns].mean()
cluster_profiles

,goals_per90_percentile,assists_per90_percentile,xG_per90_percentile,xAG_per90_percentile,progressive_passes_per90_percentile,progressive_carries_per90_percentile,dribbles_successful_per90_percentile,shot_creating_actions_per90_percentile,goal_creating_actions_per90_percentile,tackles_per90_percentile,blocks_per90_percentile
cluster,,,,,,,,,,,
0,0.823129,0.741497,0.755102,0.612245,0.721088,0.551020,0.721088,0.748299,0.809524,0.605442,0.605442
1,0.374150,0.319728,0.380952,0.278912,0.285714,0.292517,0.238095,0.251701,0.319728,0.544218,0.374150
2,0.374150,0.510204,0.435374,0.680272,0.564626,0.727891,0.612245,0.571429,0.442177,0.421769,0.591837


## Generalizing clustering across position groups

In [24]:
def find_optimal_k(data, k_range=range(2, 7), min_improvement=0.10):
    """
    Tests each K in k_range, and returns the smallest K where adding
    another cluster no longer improves inertia by at least
    `min_improvement` (as a fraction of the previous inertia).
    """
    inertias = []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(data)
        inertias.append(kmeans.inertia_)
    
    for i in range(1, len(inertias)):
        improvement = (inertias[i-1] - inertias[i]) / inertias[i-1]
        if improvement < min_improvement:
            return list(k_range)[i-1]  # previous K was the elbow point
    
    return list(k_range)[-1]  # fallback: use the largest K tested

In [25]:
def cluster_position_group(df, position_group, feature_columns, max_k=6, min_players_per_cluster=5):
    """
    Runs the full clustering pipeline for a single position group:
    finds optimal K, fits K-Means, and returns the group with cluster labels.
    """
    group_df = df[df['position_group'] == position_group].copy()
    n_players = group_df.shape[0]
    
    # cap max K so clusters aren't too small to be meaningful
    k_upper_bound = min(max_k, n_players // min_players_per_cluster)
    if k_upper_bound < 2:
        group_df['cluster'] = 0  # too few players to cluster meaningfully
        return group_df
    
    k = find_optimal_k(group_df[feature_columns], k_range=range(2, k_upper_bound + 1))
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    group_df['cluster'] = kmeans.fit_predict(group_df[feature_columns])
    
    return group_df

In [27]:
position_groups_list = ['Centre-Back', 'Full-Back', 'Defensive Midfielder',
                          'Central Midfielder', 'Attacking Midfielder', 'Winger', 'Centre-Forward']

clustered_groups = []
for group in position_groups_list:
    clustered = cluster_position_group(df, group, percentile_columns)
    clustered_groups.append(clustered)

df_clustered = pd.concat(clustered_groups, ignore_index=True)
df_clustered[['player_name', 'position_group', 'cluster']].groupby(['position_group', 'cluster']).size()

position_group        cluster
Attacking Midfielder  0           6
                      1           5
                      2           4
                      3           6
Central Midfielder    0           5
                      1           6
                      2           6
Centre-Back           0          11
                      1          10
                      2           9
                      3          10
Centre-Forward        0           5
                      1           3
                      2           6
                      3           7
Defensive Midfielder  0           3
                      1           3
                      2           8
                      3           7
Full-Back             0           8
                      1          10
                      2           7
                      3          10
Winger                0           9
                      1          12
                      2           5
dtype: int64

Note: for Centre-Forward, the automatic elbow detection selected K=4,
slightly different from the K=3 chosen by visual inspection earlier in
this notebook. This is expected — the automatic method uses a fixed
10% improvement threshold, while the manual choice involved some
judgment. Both are reasonable; the automatic version is used for
consistency across all position groups.

In [28]:
cluster_profiles_all = df_clustered.groupby(['position_group', 'cluster'])[percentile_columns].mean()
cluster_profiles_all

goals_per90_percentile  \
position_group       cluster                           
Attacking Midfielder 0                      0.753968   
                     1                      0.457143   
                     2                      0.380952   
                     3                      0.444444   
Central Midfielder   0                      0.435294   
                     1                      0.774510   
                     2                      0.362745   
Centre-Back          0                      0.471591   
                     1                      0.595000   
                     2                      0.508333   
                     3                      0.478750   
Centre-Forward       0                      0.466667   
                     1                      0.888889   
                     2                      0.666667   
                     3                      0.285714   
Defensive Midfielder 0                      0.619048   
                     1                      0.888889   
                     2                      0.580357   
                     3                      0.261905   
Full-Back            0                      0.262500   
                     1                      0.618571   
                     2                      0.430612   
                     3                      0.670000   
Winger               0                      0.459402   
                     1                      0.692308   
                     2                      0.211538   

                              assists_per90_percentile  xG_per90_percentile  \
position_group       cluster                                                  
Attacking Midfielder 0                        0.809524             0.642857   
                     1                        0.600000             0.419048   
                     2                        0.250000             0.511905   
                     3                        0.357143             0.500000   
Central Midfielder   0                        0.764706             0.447059   
                     1                        0.519608             0.774510   
                     2                        0.343137             0.352941   
Centre-Back          0                        0.765909             0.559091   
                     1                        0.487500             0.705000   
                     2                        0.350000             0.458333   
                     3                        0.405000             0.317500   
Centre-Forward       0                        0.257143             0.457143   
                     1                        0.777778             0.761905   
                     2                        0.619048             0.730159   
                     3                        0.523810             0.292517   
Defensive Midfielder 0                        0.333333             0.396825   
                     1                        0.444444             0.904762   
                     2                        0.821429             0.642857   
                     3                        0.299320             0.278912   
Full-Back            0                        0.796429             0.292857   
                     1                        0.282857             0.634286   
                     2                        0.293878             0.334694   
                     3                        0.674286             0.697143   
Winger               0                        0.307692             0.465812   
                     1                        0.650641             0.663462   
                     2                        0.584615             0.269231   

                              xAG_per90_percentile  \
position_group       cluster                         
Attacking Midfielder 0                    0.753968   
                     1                    0.685714   
                     2          

## Exporting final dataset

In [29]:
df_clustered.to_csv('../data/processed/players_clustered_2024.csv', index=False)

## Conclusion

- Ran K-Means clustering separately within each of the 7 position groups,
  using within-position percentile ranks of performance metrics as features
- Automated K selection via a simplified elbow method (stop when
  improvement in inertia falls below 10%), capped to ensure at least 5
  players per cluster given small group sizes
- Resulting clusters show visually distinct profiles per position (e.g.
  Centre-Forward Cluster 1: elite all-round finishers; Winger Cluster 2:
  build-up focused wide players with low finishing output) — providing a
  first-pass approximation of "playing style" beyond raw position labels
- Known limitation: some clusters (e.g. Centre-Forward Cluster 0, Central
  Midfielder Cluster 2) appear to separate more by overall production
  volume than by distinct style — a common K-Means limitation on a small,
  correlated feature set
- Manually naming/labeling each cluster's style is a natural next step,
  well suited for the Power BI dashboard
- Result exported to data/processed/players_clustered_2024.csv